In [ ]:
# *_readout_time.bin = datalist 
    ## chemdata size: P = nrows x ncols --> datalist[0:P-1]
    ## metatadata size: A --> datalist[P]
    ## time: t --> datalist[P+1]
    ## 1 frame size = chemdata + metadata =  P + A
    ## number of frames = number of time = len(datalist)/(P+A)

# idx_active = (idx_active_input == 400) & idx_active_temp & idx_active_gain & idx_active_lin
    ## idx_active_list 3D from idx_cative_list from *find_active*.bin file
    ## idx_active_temp set center of 5x5 false because it's a temperature
    ## idx_active_gain = idx_active_vref & idx_active_vrange
        ### delta_threshold 50
        ### range_threshold 10
        ### idx_active_vref: gain [:, :, 1] - [:, :, 3] > delta_threshold
        ### idx_active_vrange: min(gain)+range_threshold <= gain [:, :, 4] <= max(gain) - range_threshold
    ## idx_active_lin = np.all(frame_3d_lin != 0, axis=2)

# time_npr                  # from *_readout_time.bin
# well_3d_npr               # from *_readout_time.bin
# well_3d_temp_npr          # from well_3d_npr, get the center of 5x5 frame, for temp --> also for start, settled, end idx
# well_3d_gain              # nrow x ncol x 8, from *gain*.bin file --> for linearise params A B C D (0, 1, 2, 3)
# well_3d_lin               # Only linearise data which its gain is active, A > 300, B > 1, and X > C -> linearised = -1/B * log ((X-C)/A) + D
# idx_start                 # from temp (mean, 1D), idx_start: temp > temp[0] + (mean *0.9)
# idx_settled               # from temp (mean, 1D), idx_settled = idx_start + idx(chem_diff of first 50 > 20) + 3 (or = idx_start if no chem_diff > 20)
# idx_end                   # from temp (mean, 1D), idx_end: last of (start + 0.95var < temp < start + 1.05var )
# idx_active                # active pixels id

# time                      # sliced time_npr from time_settled to time_end, substracted by time_start
# time_min                  # time in minute
# well_nrows                # number of well rows
# well_ncols                # number of well columns
# well_temp_nrows           # number of temperature rows
# well_temp_ncols           # number of temperature columns

# well_3d_npr               # 3D chemical (nrow x ncol x T)
# well_3d_nl                # 3D chemical sliced by settled time to end time (nrow x ncol x Tsliced)
# well_3d_lin               # linearised well_3d_npr
# well_3d                   # well_3d_lin sliced by settled time to end time (nrow x ncol x Tsliced)

# well_2d_npr               # converted 3D chemical into 2D (T x (nrow x ncol)) --> 1 row = 1 time for all pixels
# well_2d_nl                # converted sliced 3D chemical to 2D (Tsliced x (nrow x ncol))
# well_2d_nl_bs             # 2D chemical substracted by initial chemical value (Tsliced x (nrow x ncol))
# well_2d_nl_bs_active      # well_2d_nl_bs filtered by active pixels (Tsliced x active_pixels)
# well_2d_nl_bs_active_mean # mean of well_2d_nl_bs_active (Tsliced x 1)
# well_2d_nl_active         # well_2d_nl active pixels (Tsliced x (nrow x ncol)) not substracted by bs
# well_2d_nl_active_mean    # mean of well_2d_nl_active (Tsliced x 1)
# well_2d                   # linearised 3D to 2D (Tsliced x (nrow x ncol))
# well_2d_bs                # well_2d substracted by its initial value (Tsliced x (nrow x ncol))
# well_2d_active            # well_2d filtered by active pixels (Tsliced x active_pixels)
# well_2d_bs_active         # well_2d_bs filtered by active pixels (Tsliced x active_pixels)
# well_2d_bs_active_mean    # mean of well_2d_bs_active (Tsliced x 1)
 

# well_3d_temp_npr          # 3D temperature (nrow x ncol x T)
# well_2d_temp_npr          # converted 3D temperature into 2D (T x (nrow x ncol)) --> 1 row = 1 time for all pixels
# well_temp_lin2d           # 2D temperature in AC form (linearised by [10b ln((x-c)/a)] and substracted by the 1st temp value) (T x (nrow x ncol))
# well_temp_mean_then_lin   # mean of all well_temp_lin2d nrow x ncol (T x 1)
# well_temp_2D_NEW          # 2D temperature in AC form (linearised by [(-1/b ln((x-c)/a))+d]) (T x (nrow x ncol))
# well_temp_mean_NEW        # mean of all well_temp_mean_NEW nrow x ncol (T x 1)


# Readout + processing notes (Titan: `nrows=290`, `ncols=204`)

## 1) Readout binary layout (`*_readout_time.bin` → `data_list`)
Let:

- `P = nrows * ncols` = number of chemical pixels per frame  
- `A` = number of metadata values per frame  
- `N = P + A` = total values per frame  

**Indexing in `data_list` (per frame):**
- Chemical data (pixels): `data_list[0 : P]`  (i.e., indices `0..P-1`)
- Metadata size (stored in stream): `A = data_list[P]`
- Time (stored in stream): `t = data_list[P+1]`

**Derived quantities:**
- Frame size: `N = P + A`
- Number of frames (= number of time samples): `n_frames = len(data_list) / (P + A)`


---

## 2) Active pixel mask (`idx_active`)
Final mask definition:

```python
idx_active = (
    (idx_active_input == 400)
    & idx_active_temp
    & idx_active_gain
    & idx_active_lin
)
```

### Components
- **`idx_active_input`**
  - From `idx_active_list` (loaded from `*find_active*.bin`), reshaped/permuted into input indexing.

- **`idx_active_temp`**
  - Marks temperature pixels as inactive.
  - For Titan (5×5 blocks), the center pixel (offset `[2::5, 2::5]`) is temperature → set to `False`.

- **`idx_active_gain = idx_active_vref & idx_active_vrange`**
  - `delta_threshold = 50`
  - `range_threshold = 10`

  **(a) vref filter**
  - `idx_active_vref`:  
    `gain[:, :, 1] - gain[:, :, 3] > delta_threshold`

  **(b) vrange filter**
  - `idx_active_vrange`: keep pixels not near global min/max (avoid saturation/clipping), e.g.  
    `min(gain)+range_threshold <= gain[:, :, :4] <= max(gain)-range_threshold`
    (applied across the first 4 gain samples)

- **`idx_active_lin`**
  - Pixel must be non-zero across time after linearisation:  
    `idx_active_lin = np.all(frame_3d_lin != 0, axis=2)`


---

## 3) Core variables (inputs → derived)

### Inputs / raw
- `time_npr`  
  From `*_readout_time.bin`

- `well_3d_npr`  
  Raw chemical well data from `*_readout_time.bin`  
  Shape: `(nrows, ncols, T)`

- `well_3d_temp_npr`  
  Temperature pixels extracted from `well_3d_npr` (center of each 5×5 block).  
  Used to compute `idx_start`, `idx_settled`, `idx_end`.

- `well_3d_gain`  
  Gain file, shape `(nrows, ncols, 8)` from `*gain*.bin`  
  Used to compute linearisation parameters `A, B, C, D`.

### Linearisation (per pixel)
Linearise only if gain pixel is active and parameters are valid, e.g.
- gain is active
- `A > 300`
- `B > 1`
- `X > C`

Model (inverse exponential):
\[
\text{linearised} = -\frac{1}{B}\ln\left(\frac{X - C}{A}\right) + D
\]

- `well_3d_lin`: linearised version of `well_3d_npr`

### Experiment boundaries (indices)
- `idx_start`  
  From temperature (mean 1D):  
  `idx_start: temp > temp[0] + (mean * 0.9)`  *(as described)*

- `idx_settled`  
  From temperature + chemical change:
  `idx_settled = idx_start + idx(chem_diff > 20) + 3`  
  (or `idx_settled = idx_start` if no `chem_diff > 20`)

- `idx_end`  
  From temperature stability window:
  last index where  
  `start + 0.95*var < temp < start + 1.05*var`

- `idx_active`  
  Final active pixel mask (ID / boolean mask)


---

## 4) Time vectors (sliced)
- `time`  
  `time_npr[idx_settled:idx_end] - time_npr[idx_start]`

- `time_min`  
  Time in minutes

---

## 5) Shapes / dimensions
- `well_nrows`, `well_ncols`  
  Well chemical dimensions

- `well_temp_nrows`, `well_temp_ncols`  
  Well temperature dimensions


---

## 6) Chemical representations (3D → 2D, slicing, baseline subtraction)

### 3D
- `well_3d_npr`  
  Raw chemical: `(nrows, ncols, T)`

- `well_3d_nl`  
  Raw chemical sliced settled→end: `(nrows, ncols, T_sliced)`

- `well_3d_lin`  
  Linearised chemical (full time): `(nrows, ncols, T)`

- `well_3d`  
  Linearised chemical sliced settled→end: `(nrows, ncols, T_sliced)`

### 2D (time-major)
- `well_2d_npr`  
  `well_3d_npr → (T, nrows*ncols)`  
  (each row is one time sample, across all pixels)

- `well_2d_nl`  
  Sliced raw: `(T_sliced, nrows*ncols)`

- `well_2d_nl_bs`  
  Baseline-subtracted raw: `(T_sliced, nrows*ncols)`

- `well_2d_nl_bs_active`  
  Active pixels only: `(T_sliced, n_active_pixels)`

- `well_2d_nl_bs_active_mean`  
  Mean across active pixels: `(T_sliced,)` or `(T_sliced, 1)`

- `well_2d_nl_active`  
  Raw sliced, active pixels (no baseline subtraction)

- `well_2d_nl_active_mean`  
  Mean across active pixels (raw, no BS)

- `well_2d`  
  Linearised sliced: `(T_sliced, nrows*ncols)`

- `well_2d_bs`  
  Baseline-subtracted linearised: `(T_sliced, nrows*ncols)`

- `well_2d_active`  
  Linearised active pixels: `(T_sliced, n_active_pixels)`

- `well_2d_bs_active`  
  Linearised + baseline-subtracted active pixels: `(T_sliced, n_active_pixels)`

- `well_2d_bs_active_mean`  
  Mean across active pixels: `(T_sliced,)` or `(T_sliced, 1)`


---

## 7) Temperature representations
- `well_3d_temp_npr`  
  Temperature 3D: `(n_temp_rows, n_temp_cols, T)`

- `well_2d_temp_npr`  
  Temperature 3D → 2D: `(T, n_temp_rows*n_temp_cols)`

### Temperature linearisation variants
- `well_temp_lin2d`  
  AC-form temperature transform:  
  \[
  10b\ln\left(\frac{x-c}{a}\right) \ \text{then subtract the first temp value}
  \]
  Shape: `(T, n_temp_pixels)`

- `well_temp_mean_then_lin`  
  Mean across all temperature pixels after linearisation: `(T,)` or `(T, 1)`

- `well_temp_2D_NEW`  
  Alternative inverse-exponential transform:
  \[
  \left(-\frac{1}{b}\ln\left(\frac{x-c}{a}\right)\right) + d
  \]
  Shape: `(T, n_temp_pixels)`

- `well_temp_mean_NEW`  
  Mean across all pixels of `well_temp_2D_NEW`: `(T,)` or `(T, 1)`